# Projet NLP : Assistant d'onboarding RH intelligent

**Auteurs :** Arthur DANJOU, Aksscel Meh-Rik, Moritz von SIEMENS | **Cours :** Natural Language Processing — M2

## Contexte

Ce projet intègre les concepts des **5 TPs** du cours pour construire un assistant conversationnel RH intelligent capable d'accompagner un nouvel employé pendant sa première semaine chez **TechCorp**.

L'assistant doit :
- **Répondre à des questions** en s'appuyant sur une base de connaissances interne (RAG — TP1/TP5)
- **Accéder à des données structurées** via une interface standard (TP3/TP4)
- **Déclencher des actions** au moyen d'outils : planifier des réunions, poser des congés… (TP4)
- **Analyser la tokenisation** des documents RH (TP2)

### Architecture

```
┌──────────────────────────────────────────────────────────┐
│           Assistant d'Onboarding RH — TechCorp           │
│                                                          │
│  📝 Prompts systèmes (TP3 — LangChain LCEL)             │
│  🧠 Mémoire conversationnelle à fenêtre glissante (TP4) │
│  🔧 Outils :                                            │
│     ├── 🔍 Recherche base de connaissances (TP5 — RAG)  │
│     ├── 👤 Annuaire des employés                        │
│     ├── 📅 Planification de réunions                    │
│     ├── 🏖️ Demande de congés                            │
│     └── 🕐 Date et heure actuelle                       │
│  🔄 Boucle ReAct : reason → act → observe (TP4)         │
│  📊 Embeddings MistralAI + Qdrant Vector Store (TP1/TP5)│
│  🔤 Analyse de tokenisation BPE (TP2)                   │
└──────────────────────────────────────────────────────────┘
```

In [ ]:
import os
import re
from datetime import UTC, datetime
from pathlib import Path
from uuid import uuid4

import tiktoken
from langchain.agents import create_agent
from langchain_core.documents import Document
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

import pandas as pd

os.environ["MISTRAL_API_KEY"] = ""

DATA_DIR = Path("data")

print("✅ Imports et configuration terminés!")


✅ Imports et configuration terminés!


## Partie 1 : Analyse de tokenisation (TP2)

Avant de construire notre assistant, analysons comment un LLM tokenise les documents RH. Le TP2 nous a appris que la tokenisation BPE (Byte Pair Encoding) a un impact direct sur le coût et la qualité des requêtes — en particulier, le **français utilise plus de tokens** que l'anglais pour un même contenu sémantique.

In [6]:
# Analyse de tokenisation des documents RH (TP2)
encoding = tiktoken.get_encoding("cl100k_base")

# Chargement des documents RH depuis data/entreprise.md
entreprise_md = (DATA_DIR / "entreprise.md").read_text(encoding="utf-8")

# Découpage par sections markdown (## Titre)
sections = re.split(r"\n## ", entreprise_md)
hr_documents = []
categories = []
for section in sections[1:]:  # skip le titre principal
    lines = section.strip().split("\n", 1)
    category = lines[0].strip().lower()
    content = lines[1].strip() if len(lines) > 1 else ""
    categories.append(category)
    hr_documents.append(content)

print(f"📂 {len(hr_documents)} sections chargées depuis data/entreprise.md\n")

# Fonctions utilitaires de tokenisation (TP2)
def count_tokens(text: str, enc: tiktoken.Encoding = encoding) -> int:
    """Compte le nombre de tokens dans un texte."""
    return len(enc.encode(text))

def visualize_tokens(text: str, enc: tiktoken.Encoding = encoding) -> None:
    """Visualise les premiers tokens d'un texte."""
    tokens = enc.encode(text)
    print(f"Texte: '{text[:70]}...'")
    print(f"Nombre de tokens: {len(tokens)}")
    for i, token in enumerate(tokens[:10], 1):
        decoded = enc.decode([token]).replace("\n", "\\n")
        print(f"  Token {i}: {token:6d} → '{decoded}'")
    if len(tokens) > 10:
        print(f"  ... ({len(tokens) - 10} tokens restants)")
    print()

# Analyse de chaque document
print("📊 Analyse de tokenisation des documents RH\n" + "=" * 55)
total_tokens = 0
for i, doc in enumerate(hr_documents, 1):
    tc = count_tokens(doc)
    total_tokens += tc
    print(f"Doc {i:2d} ({categories[i-1]:15s}): {tc:4d} tokens | {len(doc):4d} chars | ratio: {len(doc)/tc:.1f} chars/token")

print(f"\n📈 Total: {total_tokens} tokens pour {len(hr_documents)} documents")
print(f"   Moyenne: {total_tokens / len(hr_documents):.0f} tokens/document")

# Comparaison FR vs EN (TP2: le français est plus coûteux en tokens)
fr_text = "Bienvenue chez TechCorp! Notre entreprise est spécialisée dans le développement de logiciels."
en_text = "Welcome to TechCorp! Our company specializes in software development."
print("\n🌍 Comparaison FR vs EN (TP2):")
print(f"   FR: {count_tokens(fr_text)} tokens → '{fr_text}'")
print(f"   EN: {count_tokens(en_text)} tokens → '{en_text}'")
print("   → Le français consomme plus de tokens que l'anglais!")

# Visualisation détaillée d'un document
print("\n🔍 Détail de tokenisation:")
visualize_tokens(hr_documents[4])


📂 10 sections chargées depuis data/entreprise.md

📊 Analyse de tokenisation des documents RH
Doc  1 (accueil        ):   43 tokens |  174 chars | ratio: 4.0 chars/token
Doc  2 (politique de congés):   50 tokens |  169 chars | ratio: 3.4 chars/token
Doc  3 (procédures d'onboarding):   54 tokens |  181 chars | ratio: 3.4 chars/token
Doc  4 (équipe it      ):   51 tokens |  170 chars | ratio: 3.3 chars/token
Doc  5 (avantages sociaux):   48 tokens |  185 chars | ratio: 3.9 chars/token
Doc  6 (télétravail    ):   45 tokens |  175 chars | ratio: 3.9 chars/token
Doc  7 (formation continue):   40 tokens |  165 chars | ratio: 4.1 chars/token
Doc  8 (horaires de travail):   55 tokens |  177 chars | ratio: 3.2 chars/token
Doc  9 (sécurité       ):   51 tokens |  187 chars | ratio: 3.7 chars/token
Doc 10 (restauration   ):   50 tokens |  169 chars | ratio: 3.4 chars/token

📈 Total: 487 tokens pour 10 documents
   Moyenne: 49 tokens/document

🌍 Comparaison FR vs EN (TP2):
   FR: 20 tokens → 'Bienv

## Partie 2 : Base de connaissances vectorielle (TP1 + TP5)

On crée une base de connaissances en utilisant les **embeddings MistralAI** (TP1) et un **vector store Qdrant** (TP5). Chaque document RH est transformé en vecteur dense, permettant une recherche sémantique par similarité cosinus.

In [7]:
# Création de la base de connaissances vectorielle (TP1: Embeddings + TP5: RAG)

# Création des Document objects avec métadonnées (pattern TP5)
categories = [
    "accueil", "congés", "onboarding", "équipe", "avantages",
    "télétravail", "formation", "horaires", "sécurité", "restauration",
]

documents = []
for i, text in enumerate(hr_documents):
    doc = Document(
        page_content=text,
        metadata={"source": "TechCorp HR", "doc_id": f"hr-{i+1}", "category": categories[i]},
    )
    documents.append(doc)

# Initialisation des embeddings MistralAI (TP5)
embeddings = MistralAIEmbeddings(model="mistral-embed")

# Création du vector store Qdrant en mémoire (TP5)
COLLECTION_NAME = "techcorp-hr"
EMBEDDING_SIZE = len(embeddings.embed_query("test"))

qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_SIZE, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=qdrant_client,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
)

# Indexation des documents dans le vector store
ids = [str(uuid4()) for _ in range(len(documents))]
vector_store.add_documents(documents=documents, ids=ids)

print(f"✅ {len(documents)} documents indexés dans Qdrant")
print(f"   Dimension des embeddings: {EMBEDDING_SIZE}")
print("   Métrique de distance: Cosine similarity")

# Test de recherche sémantique (TP1: cosine similarity + TP5: retrieval)
print("\n🔍 Test de recherche sémantique:")
test_queries = ["jours de congé", "télétravail", "manger midi"]
for query in test_queries:
    results = vector_store.similarity_search(query, k=2)
    print(f"\n  Query: '{query}'")
    for _j, res in enumerate(results, 1):
        print(f"    → [{res.metadata['category']}] {res.page_content[:80]}...")


/Users/arthurdanjou/Workspace/NLP-Intelligent HR Onboarding Assistant with RAG and LangChain/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 10 documents indexés dans Qdrant
   Dimension des embeddings: 1024
   Métrique de distance: Cosine similarity

🔍 Test de recherche sémantique:

  Query: 'jours de congé'
    → [congés] Les employés ont droit à 25 jours de congé payés par an. Les RTT sont calculés a...
    → [télétravail] La politique de télétravail autorise 3 jours par semaine maximum. Les mardis et ...

  Query: 'télétravail'
    → [télétravail] La politique de télétravail autorise 3 jours par semaine maximum. Les mardis et ...
    → [accueil] Bienvenue chez TechCorp! Notre entreprise est spécialisée dans le développement ...

  Query: 'manger midi'
    → [restauration] La cantine d'entreprise est ouverte de 11h30 à 14h au rez-de-chaussée. Un espace...
    → [horaires] Les horaires sont flexibles entre 8h et 19h, avec une plage fixe de 10h à 16h. L...


## Partie 3 : Configuration du LLM et données structurées (TP3)

On initialise le modèle **ChatMistralAI** (TP3) et on prépare les données structurées des employés. Le TP3 nous a montré comment configurer un LLM via LangChain avec des messages et des chaînes LCEL.

In [8]:
# Chargement des données employés depuis data/employés.json
import json

with (DATA_DIR / "employés.json").open(encoding="utf-8") as f:
    employee_data = json.load(f)

df_employees = pd.DataFrame(employee_data)
print("👥 Annuaire des employés TechCorp (chargé depuis data/employés.json):\n")
print(df_employees.to_string(index=False))

# Initialisation du modèle LLM (TP3: ChatMistralAI)
model = ChatMistralAI(model="mistral-small-latest", temperature=0)

# Test du modèle (TP3: invoke avec messages)
response = model.invoke("Dis bonjour en une phrase, en français!")
print(f"\n🤖 Test du modèle: {response.content}")


👥 Annuaire des employés TechCorp (chargé depuis data/employés.json):

         nom              poste departement               email bureau      telephone
Alice Martin Développeur Senior          IT  alice@techcorp.com  3A-12 01 23 45 67 01
  Bob Durand     UI/UX Designer      Design    bob@techcorp.com  2B-05 01 23 45 67 02
Claire Petit         Manager IT          IT claire@techcorp.com  4A-01 01 23 45 67 03
David Moreau    DevOps Engineer          IT  david@techcorp.com  3A-14 01 23 45 67 04
   Eva Leroy     Data Scientist        Data    eva@techcorp.com  3B-08 01 23 45 67 05

🤖 Test du modèle: "Bonjour, comment ça va aujourd'hui ?" 😊


## Partie 4 : Définition des outils (TP4)

Le TP4 nous a appris à définir des **outils** (tools) avec le décorateur `@tool` de LangChain. L'agent pourra invoquer ces outils de manière autonome via la boucle **ReAct** : il raisonne sur la question, choisit l'outil adapté, et observe le résultat avant de répondre.

In [9]:
# Définition des outils de l'assistant (TP4: @tool decorator + tool calling)

@tool
def search_knowledge_base(query: str) -> str:
    """Recherche dans la base de connaissances RH de TechCorp.

    Args:
        query: La question ou le sujet à rechercher

    """
    results = vector_store.similarity_search(query, k=3)
    if not results:
        return "Aucun document pertinent trouvé."
    formatted = []
    for _i, doc in enumerate(results, 1):
        formatted.append(f"[{doc.metadata['category']}] {doc.page_content}")
    return "\n\n".join(formatted)

@tool
def get_employee_info(name: str) -> str:
    """Récupère les informations d'un employé par son nom.

    Args:
        name: Le nom ou partie du nom de l'employé

    """
    # Recherche insensible à la casse dans la colonne Nom
    name_col = df_employees.columns[0]
    result = df_employees[df_employees[name_col].str.contains(name, case=False, na=False)]
    if not result.empty:
        return result.to_string(index=False)
    return f"Aucun employé trouvé pour '{name}'."

@tool
def schedule_meeting(participants: str, date: str, subject: str) -> str:
    """Planifie une réunion avec des participants.

    Args:
        participants: Noms des participants séparés par des virgules
        date: Date et heure de la réunion
        subject: Sujet de la réunion

    """
    return (
        f"✅ Réunion planifiée!\n"
        f"  Sujet: {subject}\n"
        f"  Participants: {participants}\n"
        f"  Date: {date}\n"
        f"  📧 Invitations envoyées."
    )

@tool
def submit_leave_request(employee_name: str, start_date: str, end_date: str, leave_type: str) -> str:
    """Soumet une demande de congé.

    Args:
        employee_name: Nom de l'employé
        start_date: Date de début du congé
        end_date: Date de fin du congé
        leave_type: Type de congé (congé payé, RTT, maladie)

    """
    return (
        f"✅ Demande de congé enregistrée!\n"
        f"  Employé: {employee_name}\n"
        f"  Type: {leave_type}\n"
        f"  Du {start_date} au {end_date}\n"
        f"  ⏳ En attente de validation."
    )

@tool
def get_current_time() -> str:
    """Retourne la date et l'heure actuelles."""
    now = datetime.now(tz=UTC)
    return f"Date: {now.strftime('%d/%m/%Y')} | Heure: {now.strftime('%H:%M:%S')} UTC"

tools = [search_knowledge_base, get_employee_info, schedule_meeting, submit_leave_request, get_current_time]
print("✅ 5 outils définis:")
for t in tools:
    print(f"   🔧 {t.name}: {t.description.splitlines()[0]}")


✅ 5 outils définis:
   🔧 search_knowledge_base: Recherche dans la base de connaissances RH de TechCorp.
   🔧 get_employee_info: Récupère les informations d'un employé par son nom.
   🔧 schedule_meeting: Planifie une réunion avec des participants.
   🔧 submit_leave_request: Soumet une demande de congé.
   🔧 get_current_time: Retourne la date et l'heure actuelles.


## Partie 5 : Agent conversationnel avec mémoire (TP3 + TP4)

On assemble le tout en un **agent ReAct** créé via `create_agent` de LangGraph (TP4). L'agent est enveloppé dans une classe `OnboardingAssistant` qui gère la **mémoire conversationnelle** à fenêtre glissante (pattern `SlidingWindowChatBot` du TP4), permettant des échanges multi-tours cohérents.

In [10]:
# Création de l'agent ReAct avec mémoire (TP4: LangGraph + ChatBot)

SYSTEM_PROMPT = """Tu es l'assistant d'onboarding RH de TechCorp, une entreprise de développement logiciel de 200 employés.

Ton rôle est d'accompagner les nouveaux employés pendant leur première semaine.

Tu dois:
- Répondre aux questions en recherchant dans la base de connaissances RH
- Fournir les coordonnées des collègues via l'annuaire
- Planifier des réunions d'intégration
- Aider avec les demandes de congés
- Donner la date/heure si demandé

Règles:
1. Réponds TOUJOURS en français
2. Utilise les outils pour donner des réponses précises et factuelles
3. Si l'info n'est pas disponible, dis-le clairement
4. Sois accueillant et professionnel"""

# Création de l'agent LangGraph ReAct (TP4: create_agent)
agent = create_agent(
    model=ChatMistralAI(model="mistral-small-latest", temperature=0),
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)


# Assistant avec mémoire conversationnelle (TP4: ChatBot + SlidingWindow)
class OnboardingAssistant:
    """Assistant d'onboarding avec mémoire à fenêtre glissante (TP4)."""

    def __init__(self, max_messages: int = 20) -> None:
        """Initialise l'assistant avec une mémoire limitée."""
        self.agent = agent
        self.history = []
        self.max_messages = max_messages

    def chat(self, user_message: str, verbose: bool = False) -> str:  # noqa: FBT002
        """Envoie un message et retourne la réponse."""
        self.history.append(("user", user_message))

        # Fenêtre glissante pour limiter le contexte (TP4: SlidingWindowChatBot)
        context = self.history[-self.max_messages :]
        response = self.agent.invoke({"messages": context})

        # Extraire la réponse finale de l'agent
        final_message = response["messages"][-1].content
        self.history.append(("assistant", final_message))

        if verbose:
            print("  📜 Trace:")
            for msg in response["messages"]:
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"     🔧 {tc['name']}({tc['args']})")
                elif isinstance(msg, ToolMessage):
                    print(f"     📋 → {msg.content[:100]}...")
            print()

        return final_message

    def reset(self) -> None:
        """Réinitialise la mémoire."""
        self.history = []
        print("🔄 Mémoire réinitialisée.\n")


assistant = OnboardingAssistant()
print("✅ Agent d'onboarding créé!")
print("   Pattern: ReAct (reason → act → observe)")
print("   Mémoire: Fenêtre glissante (max 20 messages)")
print(f"   Outils: {len(tools)}")


✅ Agent d'onboarding créé!
   Pattern: ReAct (reason → act → observe)
   Mémoire: Fenêtre glissante (max 20 messages)
   Outils: 5


## Partie 6 : Démonstrations

### Scénario 1 — Questions sur l'entreprise (RAG)
L'assistant utilise la recherche sémantique dans la base de connaissances pour répondre à des questions factuelles sur TechCorp.

In [11]:
# Démonstration 1 : Questions RAG sur l'entreprise (TP5)
print("=" * 65)
print("🔍 Scénario 1 : Questions sur l'entreprise (RAG)")
print("=" * 65)

questions = [
    "Combien de jours de congé ai-je par an ?",
    "Quels sont les jours de présence obligatoire au bureau ?",
    "Où est-ce que je peux déjeuner ?",
]

for q in questions:
    print(f"\n👤 {q}")
    response = assistant.chat(q, verbose=True)
    print(f"🤖 {response}")

assistant.reset()


🔍 Scénario 1 : Questions sur l'entreprise (RAG)

👤 Combien de jours de congé ai-je par an ?
  📜 Trace:
     🔧 search_knowledge_base({'query': 'nombre de jours de congé par an'})
     📋 → [congés] Les employés ont droit à 25 jours de congé payés par an. Les RTT sont calculés au prorata t...

🤖 Vous avez droit à 25 jours de congé payés par an. Les RTT sont calculés au prorata temporis.

👤 Quels sont les jours de présence obligatoire au bureau ?
  📜 Trace:
     🔧 search_knowledge_base({'query': 'jours de présence obligatoire au bureau'})
     📋 → [télétravail] La politique de télétravail autorise 3 jours par semaine maximum. Les mardis et jeudis...

🤖 Les mardis et jeudis sont des jours de présence obligatoire au bureau.

👤 Où est-ce que je peux déjeuner ?
  📜 Trace:
     🔧 search_knowledge_base({'query': "restaurant d'entreprise"})
     📋 → [restauration] La cantine d'entreprise est ouverte de 11h30 à 14h au rez-de-chaussée. Un espace déte...

🤖 La cantine d'entreprise est ouverte de 11h

### Scénario 2 — Conversation multi-tours avec mémoire et outils
L'assistant exploite sa **mémoire conversationnelle** (TP4) pour maintenir le contexte sur plusieurs échanges, et appelle des outils variés (annuaire, planification, congés) de manière autonome via la **boucle ReAct**.

In [12]:
# Démonstration 2 : Conversation multi-tours avec mémoire et outils (TP4)
print("=" * 65)
print("🧠 Scénario 2 : Conversation complète d'onboarding")
print("=" * 65)

conversation = [
    "Bonjour! Je suis Lucas, nouveau développeur chez TechCorp.",
    "Qui dirige l'équipe IT ?",
    "Peux-tu me donner ses coordonnées ?",
    "Planifie une réunion avec Claire Petit demain à 14h pour mon intégration.",
    "Quels sont mes avantages en tant que nouvel employé ?",
    "Je voudrais poser un congé du 23 au 27 décembre en congé payé.",
]

for msg in conversation:
    print(f"\n👤 {msg}")
    response = assistant.chat(msg, verbose=True)
    print(f"🤖 {response}")


🧠 Scénario 2 : Conversation complète d'onboarding

👤 Bonjour! Je suis Lucas, nouveau développeur chez TechCorp.
  📜 Trace:

🤖 Bonjour Lucas et bienvenue chez TechCorp! Je suis ravi de t'accompagner pendant ton intégration. Comment puis-je t'aider aujourd'hui?

👤 Qui dirige l'équipe IT ?
  📜 Trace:
     🔧 search_knowledge_base({'query': "qui dirige l'équipe IT"})
     📋 → [équipe] Le département IT est dirigé par Claire Petit. L'équipe compte 15 développeurs, 3 designers...

🤖 Claire Petit dirige l'équipe IT chez TechCorp.

👤 Peux-tu me donner ses coordonnées ?
  📜 Trace:
     🔧 get_employee_info({'name': 'Claire Petit'})
     📋 →          nom      poste departement               email bureau      telephone
Claire Petit Manager I...

🤖 Claire Petit, la manager de l'équipe IT, se trouve dans le bureau 4A-01. Tu peux la contacter par email à claire@techcorp.com ou par téléphone au 01 23 45 67 03.

Autre chose pour lequel je peux t'aider ?

👤 Planifie une réunion avec Claire Petit demain à

---

## 🎨 Partie 7 : Interface de Chat Interactive (TP5)

Déploiement d'une interface Gradio pour interagir avec l'assistant en mode conversationnel, en reprenant le pattern `gr.ChatInterface` du **TP5**.

In [13]:
import gradio as gr

assistant.reset()

# ── Logique de chat avec capture des outils ──────────────────────────────────

tool_log: list[list[dict]] = []  # historique des traces par tour


def chat_with_tools(message: str, _history: list) -> tuple[str, list[dict]]:
    """Chat avec capture des appels d'outils (TP5 + TP4 trace)."""
    assistant.history.append(("user", message))
    context = assistant.history[-assistant.max_messages :]
    response = assistant.agent.invoke({"messages": context})

    tool_traces = []
    for m in response["messages"]:
        if hasattr(m, "tool_calls") and m.tool_calls:
            for tc in m.tool_calls:
                tool_traces.append({"tool": tc["name"], "args": tc["args"]})  # noqa: PERF401
        elif isinstance(m, ToolMessage) and tool_traces:
            tool_traces[-1]["result"] = m.content[:300]

    final_message = response["messages"][-1].content
    assistant.history.append(("assistant", final_message))
    tool_log.append(tool_traces)
    return final_message, tool_traces


TOOL_ICONS = {
    "search_knowledge_base": "📚",
    "get_employee_info": "👤",
    "schedule_meeting": "📅",
    "submit_leave_request": "🏖️",
    "get_current_time": "🕐",
}


def format_current_trace(traces: list[dict]) -> str:
    """Format la trace du dernier tour."""
    if not traces:
        return (
            "<div style='text-align:center; padding:2em; color:#888;'>"
            "<p style='font-size:1.3em;'>🔍</p>"
            "<p>Aucun outil appelé</p></div>"
        )
    md = ""
    for i, t in enumerate(traces, 1):
        icon = TOOL_ICONS.get(t["tool"], "🔧")
        md += f"**{icon} {t['tool']}**\n\n"
        md += f"```json\n{t['args']}\n```\n\n"
        if "result" in t:
            result_preview = t["result"][:200]
            md += f"<details><summary>📋 Résultat</summary>\n\n```\n{result_preview}\n```\n\n</details>\n\n"
        if i < len(traces):
            md += "---\n\n"
    return md


def format_full_history() -> str:
    """Format l'historique complet de tous les appels d'outils."""
    if not any(tool_log):
        return "*Aucun appel d'outil enregistré.*"
    md = ""
    for turn, traces in enumerate(tool_log, 1):
        if traces:
            md += f"#### Tour {turn}\n\n"
            for t in traces:
                icon = TOOL_ICONS.get(t["tool"], "🔧")
                md += f"- {icon} **{t['tool']}**(`{t['args']}`)"
                if "result" in t:
                    md += f" → `{t['result'][:80]}…`"
                md += "\n"
            md += "\n"
    return md or "*Aucun appel d'outil enregistré.*"


def respond(message: str, chat_history: list) -> tuple[str, list, str, str, str]:
    """Gère l'envoi du message."""
    answer, traces = chat_with_tools(message, chat_history)
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": answer})
    n_tools = sum(len(t) for t in tool_log)
    stats = f"**{len(tool_log)}** tours  ·  **{n_tools}** appels d'outils"
    return "", chat_history, format_current_trace(traces), format_full_history(), stats


def reset_chat() -> tuple[list, str, str, str, str]:
    """Réinitialise tout."""
    assistant.history.clear()
    tool_log.clear()
    return (
        [],
        "<div style='text-align:center; padding:2em; color:#888;'>"
        "<p style='font-size:2em;'>💬</p>"
        "<p>Posez une question pour commencer</p></div>",
        "*Aucun appel d'outil enregistré.*",
        "**0** tours  ·  **0** appels d'outils",
        "",
    )


# ── CSS personnalisé ─────────────────────────────────────────────────────────

custom_css = """
.tool-panel {
    border-left: 3px solid var(--color-accent);
    padding-left: 12px;
}
.header-row {
    background: linear-gradient(135deg, var(--background-fill-primary), var(--background-fill-secondary));
    border-radius: 12px;
    padding: 16px 24px;
    margin-bottom: 8px;
}
footer { display: none !important; }
"""

# ── Interface ─────────────────────────────────────────────────────────────────

with gr.Blocks(
    title="TechCorp · Assistant Onboarding RH",
    fill_height=True,
) as demo:

    # En-tête
    gr.HTML(
        """
        <div style="text-align:center; padding: 1.2em 0 0.6em 0;">
            <h1 style="margin:0; font-size:1.8em;">🏢 Assistant d'Onboarding RH</h1>
            <p style="margin:4px 0 0 0; opacity:0.7; font-size:1.05em;">
                TechCorp · Entreprise, employés, congés, formations
            </p>
        </div>
        """,
    )

    with gr.Row(equal_height=True):
        # ── Colonne principale : chat ──
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Conversation",
                height=480,
                placeholder=(
                    "<div style='text-align:center; padding:3em 1em;'>"
                    "<p style='font-size:2.5em; margin:0;'>🏢</p>"
                    "<p style='font-size:1.1em; margin:8px 0 4px 0;'><strong>Bienvenue chez TechCorp !</strong></p>"
                    "<p style='opacity:0.65;'>Posez votre première question pour démarrer l'onboarding.</p>"
                    "</div>"
                ),
            )
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Votre question...",
                    label="Message",
                    submit_btn=True,
                    scale=5,
                )
                clear_btn = gr.Button("🗑️", scale=0, min_width=50)

            gr.Examples(
                examples=[
                    "Quelle est la politique de télétravail ?",
                    "Donne-moi les informations sur Alice Martin",
                    "Je souhaite poser un congé du 15 au 20 janvier",
                    "Planifie une réunion avec l'équipe Data Science demain à 14h",
                    "Quelle heure est-il ?",
                ],
                inputs=msg,
                label="💡 Suggestions",
            )

        # ── Colonne latérale : outils ──
        with gr.Column(scale=1, min_width=280):
            stats_display = gr.Markdown(
                value="**0** tours  ·  **0** appels d'outils",
            )
            with gr.Tab("Dernier appel"):
                current_tool_panel = gr.Markdown(
                    value=(
                        "<div style='text-align:center; padding:2em; color:#888;'>"
                        "<p style='font-size:2em;'>💬</p>"
                        "<p>Posez une question pour commencer</p></div>"
                    ),
                    elem_classes=["tool-panel"],
                )
            with gr.Tab("Historique"):
                history_panel = gr.Markdown(
                    value="*Aucun appel d'outil enregistré.*",
                    elem_classes=["tool-panel"],
                )

    # ── Événements ──
    msg.submit(
        respond,
        [msg, chatbot],
        [msg, chatbot, current_tool_panel, history_panel, stats_display],
    )
    clear_btn.click(
        reset_chat,
        [],
        [chatbot, current_tool_panel, history_panel, stats_display, msg],
    )

demo.launch(theme=gr.themes.Soft(), css=custom_css)


ModuleNotFoundError: No module named 'gradio'

## Conclusion

Ce projet a intégré les 5 TPs du cours de NLP dans un assistant conversationnel RH opérationnel :

| TP | Concept | Utilisation dans le projet |
|:---|:--------|:---------------------------|
| **TP1** | Embeddings | Représentation vectorielle des documents RH, similarité cosinus |
| **TP2** | Tokenisation BPE | Analyse du coût en tokens, comparaison FR/EN |
| **TP3** | LLM + LangChain | Configuration de ChatMistralAI, messages, chaînes LCEL |
| **TP4** | Agents + Mémoire | Outils `@tool`, boucle ReAct, mémoire à fenêtre glissante |
| **TP5** | RAG + Gradio | Documents vectorisés dans Qdrant, recherche sémantique, interface chat Gradio |

L'assistant **TechCorp** est capable de répondre à des questions factuelles via RAG, accéder à des données structurées, et déclencher des actions — le tout dans une conversation multi-tours avec mémoire contextuelle, exposée via une interface Gradio interactive.